# Week 34, Day 1: Spike trains as point processes: renewal models and the time-rescaling test

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w034/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Regularity of intervals, variability of counts

For gamma renewal processes with shape k = 0.5, 1, 4 and 16 and mean interval 50 ms (20 Hz), draw 8,000 intervals each (local rng, seed 34), and compute the interval CV and the Fano factor of counts in 2 s windows. Store (CV, Fano) in `res[k]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(34)
res = {}
for k in (0.5, 1, 4, 16):
    isi = r.gamma(k, 50 / k, size=8000)               # shape k, mean 50 ms
    spikes = np.cumsum(isi)
    cv = ...
    counts = np.histogram(spikes, np.arange(0, spikes[-1], 2000))[0]
    fano = ...
    res[k] = (cv, fano)
    print(f"k = {k:4}: CV {cv:.3f} (1/sqrt(k) = {1 / np.sqrt(k):.3f}), Fano in 2 s windows {fano:.3f}, CV^2 {cv ** 2:.3f}")

In [ ]:
import numpy as np
def test_cv():
    "each process's interval CV is within 0.05 of 1 / sqrt(k)"
    assert all(abs(res[k][0] - 1 / np.sqrt(k)) < 0.05 for k in res)

def test_fano():
    "each long-window Fano factor is close to CV squared"
    assert all(abs(res[k][1] - res[k][0] ** 2) < 0.3 * res[k][0] ** 2 + 0.05 for k in res)

test_cv()
test_fano()
print("All checks passed")

### Exercise: Does the model fit?

Generate 60 s of spikes from an inhomogeneous Poisson process with rate 20 + 15 sin(2πt / 1 s) Hz by thinning: draw candidate times at 35 Hz and keep each with probability λ(t)/35 Hz. Write `ks(tau)`, the KS distance between the transformed intervals z = 1 - e^(-τ) and the uniform distribution. Apply it to intervals rescaled by the true rate and by a constant rate equal to the mean.

In [ ]:
r = np.random.default_rng(2002)
T = 60000.0                                                  # ms
lam = lambda t: (20 + 15 * np.sin(2 * np.pi * t / 1000)) / 1000     # spikes per ms
Lam = lambda t: (20 * t - 15 * 1000 / (2 * np.pi) * (np.cos(2 * np.pi * t / 1000) - 1)) / 1000
cand = np.sort(r.uniform(0, T, r.poisson(35 / 1000 * T)))
s = ...                                                      # keep each candidate with probability lam / (35 / 1000)

def ks(tau):
    z = np.sort(1 - np.exp(-tau)); n = len(z)
    above = np.arange(1, n + 1) / n - z
    below = z - np.arange(0, n) / n
    return ...

print(f"{len(s)} spikes; KS distance with the true rate {ks(np.diff(Lam(s))):.3f}, with a constant rate {ks(np.diff(s) * len(s) / T):.3f}, critical value {1.36 / np.sqrt(len(s) - 1):.3f}")
for tau, label in ((np.diff(Lam(s)), "true rate"), (np.diff(s) * len(s) / T, "constant rate")):
    z = np.sort(1 - np.exp(-tau)); plt.plot(np.linspace(0, 1, len(z)), z, label=label)
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("uniform quantiles"); plt.ylabel("rescaled quantiles"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_thinning():
    "spikes were generated by thinning, so their number is close to the integral of the rate"
    assert abs(len(s) - Lam(T)) < 4 * np.sqrt(Lam(T))

def test_ks():
    "the true rate model passes the KS test and the constant-rate model fails it"
    crit = 1.36 / np.sqrt(len(s) - 1)
    assert ks(np.diff(Lam(s))) < crit and ks(np.diff(s) * len(s) / T) > crit

test_thinning()
test_ks()
print("All checks passed")